# Bank Marketing Campaign — Modeling & Evaluation


## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                              precision_score, recall_score, f1_score)

## 2. Load the Cleaned Data



In [ ]:
df = pd.read_csv('cleaned_data.csv')
print(f'Loaded: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

In [ ]:
print('Column list:', list(df.columns))
print(f'\nData types:\n{df.dtypes.value_counts()}')
print(f'\nAny missing values? {df.isnull().sum().sum()}')

In [ ]:
corr_matrix = df.corr(numeric_only=True)
target = "deposit_enc"

# ترتيب + اختيار أهم 12
sorted_corr = corr_matrix[target].abs().sort_values(ascending=False)
top_features = sorted_corr.head(12).index

plt.figure(figsize=(10, 6))
sns.heatmap(corr_matrix.loc[top_features, top_features],
            cmap='coolwarm', annot=True, fmt=".2f")

plt.title("Top Features Sorted by Correlation with Target")
plt.xticks(rotation=45)
plt.show()

## 3. Setting Up Features and Target


In [ ]:
target_col = 'deposit_enc'

# These are either the target itself or directly derived from it
drop_cols = [target_col, 'deposit_binary']

# Also drop any text columns that might have slipped through
non_numeric = df.select_dtypes(include='object').columns.tolist()
drop_cols = list(set(drop_cols + non_numeric))
drop_cols = [c for c in drop_cols if c in df.columns]

X = df.drop(columns=drop_cols)
y = df[target_col]

print(f'Features: {X.shape[1]} columns')
print(f'Target: {y.shape[0]} samples')
print(f'\nTarget balance:\n{y.value_counts()}')
print(f'\nUsing these {len(X.columns)} features:\n{list(X.columns)}')

## 4. Train/Test Split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)


## 5. Feature Scaling



In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train_unscaled = X_train.copy()
X_test_unscaled = X_test.copy()


## 6. Training & Evaluating Models


In [ ]:
# Reusable evaluation function — saves us from repeating the same code 4 times
results = []
confusion_matrices = {}

def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    """Train a model, evaluate it, print results, and store metrics."""
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    acc = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred)
    rec = recall_score(y_te, y_pred)
    f1 = f1_score(y_te, y_pred)
    cm = confusion_matrix(y_te, y_pred)
    results.append({'Model': name, 'Accuracy': acc, 'Precision': prec,
                    'Recall': rec, 'F1-Score': f1})
    confusion_matrices[name] = cm

    print(f'--- {name} ---')
    print(f'Accuracy:  {acc:.4f} ({acc*100:.2f}%)')
    print(f'Precision: {prec:.4f}  |  Recall: {rec:.4f}  |  F1: {f1:.4f}')
    print()
    print(classification_report(y_te, y_pred, target_names=['No Deposit', 'Deposit']))
    return model

### 6.1 Logistic Regression


In [ ]:
lr_model = evaluate_model(
    'Logistic Regression',
    LogisticRegression(max_iter=1000, random_state=42),
    X_train_scaled, X_test_scaled, y_train, y_test
)

### 6.2 Decision Tree


In [ ]:
dt_model = evaluate_model(
    'Decision Tree',
    DecisionTreeClassifier(max_depth=10, min_samples_split=10, random_state=42),
    X_train_unscaled, X_test_unscaled, y_train, y_test
)

### 6.3 Random Forest


In [ ]:
rf_model = evaluate_model(
    'Random Forest',
    RandomForestClassifier(n_estimators=200, max_depth=15, min_samples_split=5,
                           random_state=42 ),
    X_train_unscaled, X_test_unscaled, y_train, y_test
)

### 6.4 Support Vector Machine (SVM)


In [ ]:
svm_model = evaluate_model(
    'SVM',
    SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42),
    X_train_scaled, X_test_scaled, y_train, y_test
)

## 7. best Model




In [ ]:
results_df = pd.DataFrame(results)
best_name = results_df.loc[results_df['Accuracy'].idxmax(), 'Model']

for _, row in results_df.iterrows():
    marker = '<--- BEST' if row['Model'] == best_name else ''

    print(f"{row['Model']:25s} | "
          f"Acc: {row['Accuracy']*100:6.2f}% | "
          f"F1: {row['F1-Score']*100:6.2f}% {marker}")